In [1]:
# Import splish
import pandas as pd
import numpy as np
import re
import os
from nltk.inference.prover9 import *

os.environ["PROVER9"] = "/home/flopezp/Prover9/bin/prover9"

In [2]:
folio_full_val = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
folio_full_test = pd.read_json('/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl', lines = True)
trying_splish = pd.read_csv('/home/flopezp/Kurosagol/Ongoing/baseline_datasets/test/filtered/translation/TRANS_DeepSeek-R1-0528-Qwen3-8B.csv')
trying_splish = trying_splish.drop(columns = ["Unnamed: 0"])
trying_splish.head()

,Translation
0,['∀x (ProfessionalSoccerPlayer(x) → ¬Professio...
1,['∀x (ProfessionalSoccerPlayer(x) → ¬Professio...
2,['∀x (ProfessionalSoccerPlayer(x) → ¬Professio...
3,"['∀x (JoinSG(x) → ImproveComm(x)) ', '∀x (Stay..."
4,"['∀x (JoinSG(x) → ImproveComm(x)) ', '∀x (Stay..."


In [3]:
def prove(argument):
    goal, assumptions = argument
    g = Expression.fromstring(goal)
    alist = [Expression.fromstring(a) for a in assumptions]
    p = Prover9Command(g, assumptions=alist).prove()
    return p

# FOL to Prover9 Syntax

def switch_quantifiers(text, cuantifier):
    """
        Elimina todos los cuantificadores y los traduce a sintaxis de Prover9. Sin importar la variable ni la cantidad de apariciones. Qué pedo soy una verga para esto.

        text = str ;  Texto a modificar.
        cuantifier = str ('forall', 'exists') ; Cuantificador a modificar.
    """
    if cuantifier == 'forall':
        regex = '∀[A-z]'
        cuant = ' all '
    else:
        regex = '∃[A-z]'
        cuant = ' exists '

    owo = re.finditer(regex, text)
    aux_list = list(owo)
    if len(aux_list) == 0:
        return text

    # Redefinimos el iterador porque hacer lista de un iterador lo consume. CHINGA TU MADRE PYTHON. VETE A LA BURGER.    
    owo = re.finditer(regex, text)
    temporal_str = ''

    for _ in owo:
        if temporal_str == '':
            temporal_str = text[0:_.start()] + cuant + _.group()[-1] + ' ' + text[_.end():]
        else:
            value = re.search(regex, temporal_str)
            temporal_str = temporal_str[0:(value.start())] + cuant + value.group()[-1] + ' ' + temporal_str[value.end():]
    
    return temporal_str



def fol_to_prover9(value):
    """
        Modifica los símbolos lógicos normales y los cambia por los valores adecuados para Prover9.*
    """
    temp = value.lower()
    temp = switch_quantifiers(temp, 'forall')
    temp = switch_quantifiers(temp, 'exists')
    temp = re.sub('-', '', temp)
    temp = re.sub('¬', ' -', temp) 
    temp = re.sub('→|→', '->', temp)
    temp = re.sub('∧', '&', temp)
    temp = re.sub('∨', '|', temp)
    temp = re.sub('↔', '<->', temp)
    temp = re.sub('≠', '!=', temp)
    temp = re.sub(r'\'', '', temp)
    #temp = re.sub(r'[\'\"]', '', temp)
    #temp = re.sub(r'\[', '(', temp)
    #temp = re.sub(r'\]', ')', temp)
    temp = re.sub(r'\[', '', temp)
    temp = re.sub(r'\]', '', temp)
    temp = re.sub('∴', '', temp)
    temp = re.sub(r'(\. \()', '.(', temp)
    temp = re.sub(r'\.{2}', '.', temp)
    #temp = re.sub(r'\)\.', ')', temp)
    temp = re.sub(r'([^a-z]\.\()', '(', temp)
    temp = re.sub(r'\.', ' ', temp)
    temp = re.sub(r'\?', '', temp)
    temp = re.sub(r'\"', '', temp)
    #if '/' in temp:
    #    temp = temp[:temp.index('/')]
    #temp = temp + '.'
    return temp


def dash_predicates(text):
    """
        Cambia los predicados de la forma "texto-texto-texto(x)" -> "textotextotexto(x)"

        text = str ; el hilo a modificar.
    """
    all_values = len(re.findall(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', text))

    if all_values == 0:
        return text
    
    new_text = text
    for i in range(all_values):
        current_regex = re.search(r'[a-z0-9]+(\-[a-z0-9]+\-{0,})+[a-z0-9]+', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


def elim_spaces(text):
    """
        Elimina los espacios entre variables: lionel messi -> lionelmessi

        text = str; El texto a modificar.

        OBS: Este formato de funciones (Encontrar cantidades y luego iterar sobre las cantidades) me gusta bastante.
    """
    total_iters = len(list(re.finditer(r'([A-z]+ )+([A-z]{2,})', text)))
    if total_iters == 0:
        return text

    new_text = text
    for i in range(total_iters):
        current_regex = re.search(r'([A-z]+ )+([A-z]{2,})', new_text)
        split = current_regex.group().split()
        aux_text = ''
        for elem in split:
            aux_text = aux_text + elem
        new_text = new_text[:current_regex.start()] + aux_text + new_text[current_regex.end():]

    return new_text


# El XOR me tiene hasta los huevos cabrón te lo juro.
def xor_bonito(expression):
    """
        Elimina el símbolo de XOR, y lo reescribe en la fórmula (A OR B) AND NOT(A AND B)
    """
    individual_values = re.findall(r'([A-z|_|0-9]{2,}|¬)', expression)
    a = individual_values[0] + '(x)'
    b = individual_values[1] + '(x)'
    a_or_b = '(' + a + ' | ' + b + ')'
    not_a_and_b = ' -(' + a + ' & ' + b +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def xor_bonito_extreme(expression):
    """
        Elimina los XOR de fórmulas compuestas.
    """
    aux2 = re.search(r'([a-z]+\([a-z, ]+\)) ⊕ ([a-z]+\([a-z, ]+\))', expression)
    split = aux2.group().split('⊕')
    a_f = split[0]
    b_f = split[-1]
    a_or_b = '(' + a_f + ' | ' + b_f + ')'
    not_a_and_b = ' -(' + a_f + ' & ' + b_f +')'
    xor = a_or_b + ' & ' + not_a_and_b
    return xor

def rewrite_xor(re_search, element, comp):
    """
        Genera una nueva expresión a partir del xor bonito. 

        re_search = re.search(regex, str)
        element = str ; same str as above
        comp = bool ; True iff predicates have multiple variables.
    """
    start = re_search.start()
    end = re_search.end()

    xor_substr = element[start:end]
    xor_chido = xor_bonito_extreme(xor_substr)
    #if comp:
    #    xor_chido = xor_bonito_extreme(xor_substr)
    #else:
    #    xor_chido = xor_bonito(xor_substr)

    nuevo = element[:start] + xor_chido + element[end:]
    return nuevo


def clean(value, FOLIO):
    """
        Procesa una respuesta individual de FOLIO/GPT_TRANS/QWEN_TRANS para que se pase al formato de Prover9.
    """
    if FOLIO:
        clean_premises_aux = value.split('\n')
    else:
        clean_premises_aux = str(value).split('\',')
    clean_premises = []
    for _ in clean_premises_aux:
        if _ != '':
            clean_premises.append(_)

    if "Premises" in clean_premises[0]:
        del clean_premises[0]

    #print(clean_premises[0])

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub(r'(:::)+([ A-z.]+)', '', _)

    for _ in clean_premises:
        clean_premises[clean_premises.index(_)] = re.sub(r'[0-9]\.', '', _)
    
    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = fol_to_prover9(instance)

    #print(clean_premises[0])
    for instance in clean_premises:
        clean_premises[clean_premises.index(instance)] = elim_spaces(instance)

    try:
        # Filtro XOR sencillo
        for element in clean_premises:
            xor_count = len(re.findall('⊕', element))
            if xor_count > 0:
                value = element
                element_new = element
                while xor_count > 0:
                    aux = re.search(r'(-{0,1}[a-z]+\([a-z, ]+\)) ⊕ (-{0,1}[a-z]+\([a-z, ]+\))', element_new)
                    element_new = rewrite_xor(aux, element_new, False)
                    xor_count = len(re.findall('⊕', element_new))
                clean_premises[clean_premises.index(value)] = element_new

    except:
        #print("Hubo un Xor malo")
        a = 0    

    return clean_premises



# ========================================
# ========================================
# ========================================

def check_arg_validity(dataset, index, validation):
    """
        Dadas premisas en lenguaje lógico, verifica que la conclusión correspondiente sea TRANSerible.
    """
    if validation:
        folio_full = folio_full_val
    else:
        folio_full= folio_full_test

    can_parse, neg_parse, prover9_fatal, logical_exp = 0,0,0,0
    
    llm_value = dataset['Translation'][index]
    #folio_value = folio_full['premises-FOL'][index]

    #true_validity = folio_full['label'][index]

    #print("Logical Validity: {}".format(true_validity))

    clean_llm = clean(llm_value, False)
    #print('Clean LLM: ', clean_llm)
    #clean_folio = clean(folio_value, True)
    #print('Clean FOLIO: ', clean_folio)
    clean_conc = clean(folio_full['conclusion-FOL'][index], True) 
    #print('Clean Conc: ', clean_conc)
    
    args_llm = (
        clean_conc[0],
        clean_llm
    )

    #args_folio = (
    #    clean_conc[0],
    #    clean_folio
    #)
    
    try:
        #print("LLM Ejecutable. Valor: {}".format(prove(args_llm)))
        prove(args_llm)
        can_parse += 1
    except Exception as e:
        #print(e)
        type_error = str(repr(e).split('(')[0])
        #print(type_error)
        if type_error == 'Prover9FatalException':
            prover9_fatal+=1
        elif type_error == 'LogicalExpressionException':
            logical_exp+=1
        else:
            owo = 0
            #print('Otro error')
            #print(type_error)
            #print(e)
        neg_parse += 1
        
    # Por si queremos hacer pruebas de parsing sobre FOLIO.
    #try:
    #    print("FOLIO Ejecutable. Valor: {}".format(prove(args_folio)))
    #    fol_valid += 1
    #except Exception as e:
    #    print(e)
    #    print(type(e))
    #    fol_invalid += 1
    #print("======================")

    return can_parse, neg_parse, prover9_fatal, logical_exp
    

In [4]:
baseline_path = [
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_DeepSeek-R1-0528-Qwen3-8B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_DeepSeek-R1-Distill-Qwen-7B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gemma-3-4b-it.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gemma-3-12b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_gpt-oss-20b.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-4B-FP8.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-8B-FP8.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3-14B-FP8.csv',
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3.5-4B.csv', 
    '/home/flopezp/Kurosagol/Ongoing/baseline_datasets/{}/filtered/translation/TRANS_Qwen3.5-9B.csv'       
]

alignment_res_path = [
    '/home/flopezp/Kurosagol/Ongoing/alignment_results/{}/filtered/translation/TRANS_KTO_DeepSeek-R1-0528-Qwen3-8B.csv',
    '/home/flopezp/Kurosagol/Ongoing/alignment_results/{}/filtered/translation/TRANS_KTO_gemma-3-12b-it.csv',
    '/home/flopezp/Kurosagol/Ongoing/alignment_results/{}/filtered/translation/TRANS_KTO_Qwen3-14B.csv'
]


def evaluate_parsability(dataset_path, validation):
    if validation:
        path = dataset_path.format('validation')
        split = 'Validation'
        val = True
    else:
        path = dataset_path.format('test')
        val = False
        split = 'Test'

    dataset = pd.read_csv(path)
    try:
        dataset = dataset.drop(columns = ["Unnamed: 0"])
    except:
        excepto = 0

    parse_total, nonparse_total, prover9ex_total, logicalex_total = 0, 0, 0, 0
    for i in range(len(dataset['Translation'])):
        parse, nonparse, prover9ex, logicalex  = check_arg_validity(dataset, i, val)
        parse_total += parse
        nonparse_total += nonparse
        prover9ex_total += prover9ex
        logicalex_total += logicalex

    model_id = path.split('/')[-1][6:-4]
    length = len(dataset['Translation'])

    print("-"*60)
    print('\t {}'.format(model_id))
    print('\t Split: {}'.format(split))
    print("-"*60)
    print("Valores parseables: {}".format(parse_total))
    print("Valores NO parseables: {}".format(nonparse_total))
    print("Porcentaje parseable: {}".format(round(parse_total/length, 2)))
    print("Arity Errors: {}".format(prover9ex_total))
    print("Syntax Errors: {}".format(logicalex_total))
    print('='*60)


#parse_total, nonparse_total, prover9ex_total, logicalex_total = 0, 0, 0, 0
#for i in range(len(trying_splish['Translation'])):
#    parse, nonparse, prover9ex, logicalex  = check_arg_validity(trying_splish, i, False)
#    parse_total += parse
#    nonparse_total += nonparse
#    prover9ex_total += prover9ex
#    logicalex_total += logicalex
    

#print("Valores parseables: {}".format(parse_total))
#print("Valores NO parseables: {}".format(nonparse_total))
#print("Arity Errors: {}".format(prover9ex_total))
#print("Syntax Errors: {}".format(logicalex_total))

In [5]:
for path in baseline_path:
    evaluate_parsability(path, True)
    evaluate_parsability(path, False)

------------------------------------------------------------
	 DeepSeek-R1-0528-Qwen3-8B
	 Split: Validation
------------------------------------------------------------
Valores parseables: 97
Valores NO parseables: 106
Porcentaje parseable: 0.48
Arity Errors: 30
Syntax Errors: 69
------------------------------------------------------------
	 DeepSeek-R1-0528-Qwen3-8B
	 Split: Test
------------------------------------------------------------
Valores parseables: 123
Valores NO parseables: 103
Porcentaje parseable: 0.54
Arity Errors: 35
Syntax Errors: 67
------------------------------------------------------------
	 DeepSeek-R1-Distill-Qwen-7B
	 Split: Validation
------------------------------------------------------------
Valores parseables: 21
Valores NO parseables: 182
Porcentaje parseable: 0.1
Arity Errors: 0
Syntax Errors: 182
------------------------------------------------------------
	 DeepSeek-R1-Distill-Qwen-7B
	 Split: Test
-----------------------------------------------------

In [6]:
for path in alignment_res_path:
    evaluate_parsability(path, True)
    evaluate_parsability(path, False)

------------------------------------------------------------
	 KTO_DeepSeek-R1-0528-Qwen3-8B
	 Split: Validation
------------------------------------------------------------
Valores parseables: 88
Valores NO parseables: 115
Porcentaje parseable: 0.43
Arity Errors: 50
Syntax Errors: 62
------------------------------------------------------------
	 KTO_DeepSeek-R1-0528-Qwen3-8B
	 Split: Test
------------------------------------------------------------
Valores parseables: 123
Valores NO parseables: 103
Porcentaje parseable: 0.54
Arity Errors: 55
Syntax Errors: 48
------------------------------------------------------------
	 KTO_gemma-3-12b-it
	 Split: Validation
------------------------------------------------------------
Valores parseables: 127
Valores NO parseables: 76
Porcentaje parseable: 0.63
Arity Errors: 41
Syntax Errors: 29
------------------------------------------------------------
	 KTO_gemma-3-12b-it
	 Split: Test
------------------------------------------------------------
V